# Model, uncertainty, and inventory analysis

This notebook audits the frozen portfolio result. Model choice uses rolling validation; the final holdout is shown only as a report card.

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path('../reports/portfolio/summary.json').read_text())
summary['selected_method'], summary['series_count'], summary['holdout_start']

In [ ]:
comparison = []
for method, validation in summary['validation'].items():
    holdout = summary['holdout']['by_method'][method]
    comparison.append({
        'method': method,
        'validation_wrmsse': validation['mean_subset_wrmsse'],
        'holdout_wrmsse': holdout['subset_wrmsse'],
        'holdout_wape': holdout['wape'],
    })
sorted(comparison, key=lambda row: row['validation_wrmsse'])

In [ ]:
selected = summary['validation']['lightgbm']['folds']
seasonal = summary['validation']['seasonal_28']['folds']
fold_improvements = [
    (base['subset_wrmsse'] - model['subset_wrmsse']) / base['subset_wrmsse']
    for model, base in zip(selected, seasonal)
]
fold_improvements

In [ ]:
holdout = summary['holdout']
inventory = summary['inventory']
{
 'coverage_80': holdout['coverage_80'],
 'coverage_90': holdout['coverage_90'],
 'inventory_cost_reduction': 1 - inventory['selected']['cost'] / inventory['seasonal_28']['cost'],
 'shortage_units_reduction': inventory['seasonal_28']['shortage_units'] - inventory['selected']['shortage_units'],
}

## Decision

LightGBM clears the seasonal-baseline gate in all three validation windows, but only narrowly improves on the stronger weekday blend. The result supports a champion/challenger pilot, not automatic ordering. The inventory result is a simulation in arbitrary cost units and must be revalidated with actual stock, lead times, and service constraints.